# Distributed Spam Detection
Krzysztof Kopel, 2026

In this notebook I will try to construct (and later, deploy) an ML model to distinguish spam and normal emails. The model will be created in Ray, a library supporting distributed computing.

In [1]:
import ray
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1" # Only legacy Keras works well with Ray

In [2]:
if ray.is_initialized:
    ray.shutdown()

ray.init()

2026-06-01 00:20:08,287	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\ray\_private\worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.12.12
Ray version:,2.55.1
Dashboard:,http://127.0.0.1:8265


## Data loading and preprocessing

I will use Enron Spam dataset.

In [3]:
import pandas as pd

local_path = r"data\enron_spam_data.csv"

pdf = pd.read_csv(
    local_path,
    on_bad_lines='skip',
    encoding='utf-8',
    low_memory=False
)

pdf = pdf.fillna({"Message": "", "Subject": ""})

df = ray.data.from_pandas(pdf)
df.show(5)

2026-06-01 00:20:24,296	INFO dataset.py:3818 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
2026-06-01 00:20:24,318	INFO logging.py:416 -- Registered dataset logger for dataset dataset_1_0
2026-06-01 00:20:24,359	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_1_0. Full logs are in C:\Users\krzys\AppData\Local\Temp\ray\session_2026-06-01_00-20-00_284219_28064\logs\ray-data
2026-06-01 00:20:24,359	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_1_0: InputDataBuffer[Input] -> LimitOperator[limit=5]
2026-06-01 00:20:24,365	WARNING resource_manager.py:169 -- ⚠️  Ray's object store is configured to use only 42.9% of available memory (1.1GiB out of 2.7GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJE

{'Message ID': 0, 'Subject': 'christmas tree farm pictures', 'Message': '', 'Spam/Ham': 'ham', 'Date': '1999-12-10'}
{'Message ID': 1, 'Subject': 'vastar resources , inc .', 'Message': 'gary , production from the high island larger block a - 1 # 2 commenced on\nsaturday at 2 : 00 p . m . at about 6 , 500 gross . carlos expects between 9 , 500 and\n10 , 000 gross for tomorrow . vastar owns 68 % of the gross production .\ngeorge x 3 - 6992\n- - - - - - - - - - - - - - - - - - - - - - forwarded by george weissman / hou / ect on 12 / 13 / 99 10 : 16\nam - - - - - - - - - - - - - - - - - - - - - - - - - - -\ndaren j farmer\n12 / 10 / 99 10 : 38 am\nto : carlos j rodriguez / hou / ect @ ect\ncc : george weissman / hou / ect @ ect , melissa graves / hou / ect @ ect\nsubject : vastar resources , inc .\ncarlos ,\nplease call linda and get everything set up .\ni \' m going to estimate 4 , 500 coming up tomorrow , with a 2 , 000 increase each\nfollowing day based on my conversations with bill fis

In [4]:
def preprocess_batch(batch):
    subject_clean = batch["Subject"].fillna("")
    message_clean = batch["Message"].fillna("")

    batch["Text"] = subject_clean + " " + message_clean

    batch["Spam/Ham"] = batch["Spam/Ham"].map({"ham": 0, "spam": 1})

    return batch

processed_df = df.map_batches(preprocess_batch, batch_format="pandas")

train, test = processed_df.train_test_split(test_size=0.2, shuffle=True)
print("Train set:")
train.show(1)
print("Test set:")
test.show(1)

2026-06-01 00:20:25,272	INFO logging.py:416 -- Registered dataset logger for dataset dataset_4_0
2026-06-01 00:20:25,275	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_4_0. Full logs are in C:\Users\krzys\AppData\Local\Temp\ray\session_2026-06-01_00-20-00_284219_28064\logs\ray-data
2026-06-01 00:20:25,276	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_4_0: InputDataBuffer[Input] -> AllToAllOperator[MapBatches(preprocess_batch)->RandomShuffle]
2026-06-01 00:20:25,284	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_4_0 =======
2026-06-01 00:20:25,286	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-06-01 00:20:25,287	INFO logging_progress.py:227 -- Active & requested resources: 0/12 CPU, 0.0B/584.0MiB object store
2026-06-01 00:20:25,287	INFO logging_progress.py:181 -- 
2026-06-01 00:20:25,288	INFO logging_progress.py:231 -- MapBatches(preprocess_batch)->RandomShuffle: 0/1
2026-06-01 00:20:25,288	INFO logging_progress

Train set:
{'Message ID': 888, 'Subject': 'mary poorman interview schedule', 'Message': 'please be advised that the interview for mary poorman , to be held today will\nbe in room 3671 . the schedule is as follows :\npat clynes 2 : 30\nlauri allen 3 : 00\ngary hanks 3 : 30\ndaren farmer 4 : 00', 'Spam/Ham': 0, 'Date': '2000-06-05', 'Text': 'mary poorman interview schedule please be advised that the interview for mary poorman , to be held today will\nbe in room 3671 . the schedule is as follows :\npat clynes 2 : 30\nlauri allen 3 : 00\ngary hanks 3 : 30\ndaren farmer 4 : 00'}
Test set:
{'Message ID': 27632, 'Subject': 'i do not have anything against you', 'Message': 'ourref : cbn / go / 0 xol 2 / 05\ndate : 21 st july 2005\ntel : 234 - 1 - 470 - 7915\nemail address : mallamyusuf @ centbanks . org\ndear good friend\nafter a serious thought , i decided to reach you directly and personally\nbecause i do not have anything against you , but your nigerian partners .\ni am the director of wire 

## Training a model

In [5]:
import tensorflow as tf

def build_model() -> tf.keras.Sequential:

    return tf.keras.Sequential([
        tf.keras.Input(shape=(200,), dtype=tf.int32),
        tf.keras.layers.Embedding(input_dim=10000, output_dim=32),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(100, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

In [6]:
from ray.train.tensorflow.keras import ReportCheckpointCallback

def training_loop(config: dict):
    tf.keras.backend.clear_session()
    strategy = tf.distribute.MultiWorkerMirroredStrategy()

    batch_size = config.get("batch_size", 32)
    epochs = config.get("epochs", 4)
    X_global = config.get("X_global")
    y_global = config.get("y_global")
    X_val = config.get("X_val")
    y_val = config.get("y_val")

    with strategy.scope():
        model = build_model()
        model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    model.fit(X_global, y_global, validation_data=(X_val, y_val), batch_size=batch_size, epochs=epochs, callbacks=[ReportCheckpointCallback()])

    ray.train.report(metrics=model.evaluate(X_val, y_val, batch_size=batch_size, return_dict=True))


In [7]:
global_vectorizer = tf.keras.layers.TextVectorization(max_tokens=10000, output_sequence_length=200)
global_vectorizer.adapt(train.to_pandas()["Text"])

X_global = global_vectorizer(train.to_pandas()["Text"].astype(object)).numpy()
y_global = train.to_pandas()["Spam/Ham"].values
X_val = global_vectorizer(test.to_pandas()["Text"].astype(object)).numpy()
y_val = test.to_pandas()["Spam/Ham"].values

2026-06-01 00:20:46,563	INFO logging.py:416 -- Registered dataset logger for dataset dataset_6_2
2026-06-01 00:20:46,571	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_6_2 =======
2026-06-01 00:20:46,573	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-06-01 00:20:46,573	INFO logging_progress.py:227 -- Active & requested resources: 0/12 CPU, 0.0B/584.0MiB object store
2026-06-01 00:20:46,574	INFO logging_progress.py:192 -- ============================================
2026-06-01 00:20:46,582	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_6_2 execution finished in 0.04 seconds


2026-06-01 00:20:48,302	INFO logging.py:416 -- Registered dataset logger for dataset dataset_6_3
2026-06-01 00:20:48,311	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_6_3 =======
2026-06-01 00:20:48,313	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-06-01 00:20:48,314	INFO logging_progress.py:227 -- Active & requested resources: 0/12 CPU, 0.0B/584.0MiB object store
2026-06-01 00:20:48,314	INFO logging_progress.py:192 -- ============================================
2026-06-01 00:20:48,323	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_6_3 execution finished in 0.04 seconds
2026-06-01 00:20:49,217	INFO logging.py:416 -- Registered dataset logger for dataset dataset_6_4
2026-06-01 00:20:49,226	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_6_4 =======
2026-06-01 00:20:49,228	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-06-01 00:20:49,228	INFO logging_progress.py:227 -- Active & requested resources: 0/12 CPU, 0.0B/584.0MiB o

In [9]:
from ray.train.tensorflow import TensorflowTrainer
from ray.train import RunConfig

scaling_config = ray.train.ScalingConfig(num_workers=2, use_gpu=False)

trainer = TensorflowTrainer(
    train_loop_per_worker=training_loop,
    train_loop_config={"batch_size": 32, "epochs": 5, "X_global": X_global, "y_global": y_global, "X_val": X_val, "y_val": y_val},
    scaling_config=scaling_config,
    run_config=RunConfig(storage_path=os.path.abspath("./artifacts"))
)
results = trainer.fit()

(TrainController pid=6848) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(TrainController pid=6848) I0000 00:00:1780266134.437081    9576 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
(TrainController pid=6848) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(TrainController pid=6848) I0000 00:00:1780266136.125069    9576 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
(TrainController pid=6848) WARNING:tensorflow:From C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\tf_keras\src\losses

(RayTrainWorker pid=34160) Epoch 1/5


(RayTrainWorker pid=34160) WARNING:tensorflow:From C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\tf_keras\src\engine\base_layer_utils.py:384: The name tf.executing_eagerly_outside_functions is deprecated. Please use tf.compat.v1.executing_eagerly_outside_functions instead.
(RayTrainWorker pid=34160) 
(RayTrainWorker pid=34160) From C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\tf_keras\src\engine\base_layer_utils.py:384: The name tf.executing_eagerly_outside_functions is deprecated. Please use tf.compat.v1.executing_eagerly_outside_functions instead.
(RayTrainWorker pid=34160) 
(RayTrainWorker pid=25856) 
(RayTrainWorker pid=25856) 


(RayTrainWorker pid=34160) 
(RayTrainWorker pid=34160)   1/843 [..............................] - ETA: 10:09 - loss: 1.3848 - accuracy: 1.2500
(RayTrainWorker pid=34160)   8/843 [..............................] - ETA: 6s - loss: 1.3848 - accuracy: 1.0078   
(RayTrainWorker pid=25856) 
(RayTrainWorker pid=34160) 
(RayTrainWorker pid=25856) 
(RayTrainWorker pid=34160) 
(RayTrainWorker pid=34160)  29/843 [>.............................] - ETA: 6s - loss: 1.3777 - accuracy: 1.2220
(RayTrainWorker pid=25856) 
(RayTrainWorker pid=34160) 
(RayTrainWorker pid=25856) 
(RayTrainWorker pid=34160) 
(RayTrainWorker pid=34160)  57/843 [=>............................] - ETA: 6s - loss: 1.3508 - accuracy: 1.3882
(RayTrainWorker pid=25856) 
(RayTrainWorker pid=34160) 
(RayTrainWorker pid=25856) 
(RayTrainWorker pid=34160) 
(RayTrainWorker pid=34160) 
(RayTrainWorker pid=34160)  92/843 [==>...........................] - ETA: 5s - loss: 1.2781 - accuracy: 1.4762
(RayTrainWorker pid=25856) 
(RayTrainWorke

(RayTrainWorker pid=25856) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR [repeated 2x across cluster]
(RayTrainWorker pid=25856) I0000 00:00:1780266151.393115   25560 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`. [repeated 2x across cluster]
(RayTrainWorker pid=25856) WARNING:tensorflow:From C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\tf_keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.
(RayTrainWorker pid=25856) WARNING:tensorflow:From C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\tf_keras\src\backend.py:277: The name tf.reset_default_graph is deprecated. Please us

...

(RayTrainWorker pid=34160) Checkpoint successfully created at: Checkpoint(filesystem=local, path=C:/Users/krzys/Documents/Programy_Python/DistributedMLProject/artifacts/ray_train_run-2026-06-01_00-22-08/checkpoint_2026-06-01_00-23-03.303682) [repeated 2x across cluster]
(RayTrainWorker pid=34160) Reporting training result 4: TrainingReport(checkpoint=Checkpoint(filesystem=local, path=C:/Users/krzys/Documents/Programy_Python/DistributedMLProject/artifacts/ray_train_run-2026-06-01_00-22-08/checkpoint_2026-06-01_00-23-03.303682), metrics={'loss': 0.01303794700652361, 'accuracy': 0.9958475232124329, 'val_loss': 0.037531398236751556, 'val_accuracy': 0.9911032319068909}, validation=False) [repeated 2x across cluster]


(RayTrainWorker pid=34160) 
(RayTrainWorker pid=34160) 843/843 [==============================] - 7s 9ms/step - loss: 0.0130 - accuracy: 0.9958 - val_loss: 0.0375 - val_accuracy: 0.9911
(RayTrainWorker pid=34160) Epoch 5/5
(RayTrainWorker pid=25856) 223/843 [======>.......................] - ETA: 4s - loss: 0.0306 - accuracy: 1.9902 [repeated 9x across cluster]
(RayTrainWorker pid=25856) 247/843 [=======>......................] - ETA: 4s - loss: 0.0294 - accuracy: 1.9909 [repeated 6x across cluster]
(RayTrainWorker pid=25856) 275/843 [========>.....................] - ETA: 4s - loss: 0.0295 - accuracy: 1.9911 [repeated 7x across cluster]
(RayTrainWorker pid=25856) 303/843 [=========>....................] - ETA: 4s - loss: 0.0288 - accuracy: 1.9913 [repeated 7x across cluster]
(RayTrainWorker pid=25856) 
(RayTrainWorker pid=34160) 
(RayTrainWorker pid=25856) 
(RayTrainWorker pid=34160) 
(RayTrainWorker pid=25856) 
(RayTrainWorker pid=34160) 
(RayTrainWorker pid=25856) 332/843 [=========

(RayTrainWorker pid=34160) W0000 00:00:1780266189.866301   31972 dataset.cc:997] Input of GeneratorDatasetOp::Dataset will not be optimized because the dataset does not implement the AsGraphDefInternal() method needed to apply optimizations.
(RayTrainWorker pid=25856) Checkpoint successfully created at: Checkpoint(filesystem=local, path=C:/Users/krzys/Documents/Programy_Python/DistributedMLProject/artifacts/ray_train_run-2026-06-01_00-22-08/checkpoint_2026-06-01_00-23-03.303682)
(RayTrainWorker pid=25856) Reporting training result 4: TrainingReport(checkpoint=Checkpoint(filesystem=local, path=C:/Users/krzys/Documents/Programy_Python/DistributedMLProject/artifacts/ray_train_run-2026-06-01_00-22-08/checkpoint_2026-06-01_00-23-03.303682), metrics={'loss': 0.01303794700652361, 'accuracy': 0.9958475232124329, 'val_loss': 0.037531398236751556, 'val_accuracy': 0.9911032319068909}, validation=False)
(RayTrainWorker pid=25856) Checkpoint successfully created at: Checkpoint(filesystem=local, pat

(RayTrainWorker pid=34160) 
(RayTrainWorker pid=34160) 843/843 [==============================] - 7s 9ms/step - loss: 0.0100 - accuracy: 0.9966 - val_loss: 0.0404 - val_accuracy: 0.9913
(RayTrainWorker pid=25856) 502/843 [================>.............] - ETA: 2s - loss: 0.0213 - accuracy: 1.9927 [repeated 8x across cluster]
(RayTrainWorker pid=25856) 529/843 [=================>............] - ETA: 2s - loss: 0.0214 - accuracy: 1.9924 [repeated 8x across cluster]
(RayTrainWorker pid=25856) 556/843 [==================>...........] - ETA: 2s - loss: 0.0216 - accuracy: 1.9925 [repeated 8x across cluster]
(RayTrainWorker pid=25856) 588/843 [===================>..........] - ETA: 1s - loss: 0.0221 - accuracy: 1.9923 [repeated 10x across cluster]
(RayTrainWorker pid=25856) 615/843 [====================>.........] - ETA: 1s - loss: 0.0214 - accuracy: 1.9927 [repeated 8x across cluster]
(RayTrainWorker pid=25856) 
(RayTrainWorker pid=34160) 
(RayTrainWorker pid=25856) 643/843 [================

(RayTrainWorker pid=34160) Reporting training result 6: TrainingReport(checkpoint=None, metrics={'loss': 0.040422968566417694, 'accuracy': 0.9912514686584473}, validation=False)
(raylet) Stack (most recent call first):
(raylet)   File "<frozen importlib._bootstrap_external>", line 757 in _compile_bytecode
(raylet)   File "<frozen importlib._bootstrap_external>", line 1128 in get_code
(raylet)   File "<frozen importlib._bootstrap_external>", line 995 in exec_module
(raylet)   File "<frozen importlib._bootstrap>", line 935 in _load_unlocked
(raylet)   File "<frozen importlib._bootstrap>", line 1331 in _find_and_load_unlocked
(raylet)   File "<frozen importlib._bootstrap>", line 1360 in _find_and_load
(raylet)   File "C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\ray\dag\compiled_dag_node.py", line 28 in <module>
(raylet)   File "<frozen importlib._bootstrap>", line 488 in _call_with_frames_removed
(raylet)   File "C:\Users\krzys\Documents\Programy_

### Evaluation

In [10]:
print(f"Final results: {results.metrics}")

Final results: {'loss': 0.010032557882368565, 'accuracy': 0.9965890645980835, 'val_loss': 0.040422968566417694, 'val_accuracy': 0.9912514686584473}


Model seems to be working correctly and is ready for deployment.

In [ ]:
with open("artifacts/vocabulary.txt", "w", encoding="utf-8") as file:
    file.write("\n".join(global_vectorizer.get_vocabulary()))